In [ ]:
import sys
from pathlib import Path

# horizon/ pipeline modules use flat imports (`from fds.fd import ...`) and need
# horizon/ on sys.path; eval + package imports (`horizon.fds.fd`) need the repo
# root. Put both on the path, repo root first. See app_helpers._bootstrap_path.
REPO = Path("..").resolve()
HZN = REPO / "horizon"
for p in (str(HZN), str(REPO)):
    if p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(HZN))
sys.path.insert(0, str(REPO))

from horizon.utils.loaders import get_fds, load_table
from eval.dataset_eval import characterize_dataset
from eval.fd_eval import characterize_fds
from eval.report import characterize

In [ ]:
fds_path = Path("..") / "datasets" / "hospital_2k" / "fds.csv"
csv_path = Path("..") / "datasets" / "hospital_2k" / "clean.csv"

In [ ]:
fds = get_fds(fds_path, csv_path)
print(f"Loaded {len(fds)} FDs")
fds

In [ ]:
df = load_table(csv_path)
df.head()

In [ ]:
# drop FDs whose LHS redundancy is below MIN_REDUNDANCY — too few repeats to
# support a pattern (avg LHS group size ≈ n^MIN_REDUNDANCY).
from eval.fd_eval import fd_lhs_redundancy

MIN_REDUNDANCY = 0.1  # ~3 rows per LHS group at n≈170k

red = fd_lhs_redundancy(df, fds)
kept = [fd for fd in fds if red[fd.lhs_attributes] >= MIN_REDUNDANCY]
dropped = [fd for fd in fds if red[fd.lhs_attributes] < MIN_REDUNDANCY]
print(f"kept {len(kept)} / {len(fds)} FDs; dropping {len(dropped)} (red < {MIN_REDUNDANCY}):")
for fd in dropped:
    print(f"  {red[fd.lhs_attributes]:.3f}  {fd}")

# keep the filtered set in memory; don't overwrite the source fds.csv
fds = kept

In [ ]:
# FD-side metrics only make sense when the FDs were defined for *this* table.
# Proxy for "they belong together": both files live in the same dataset folder.
from pprint import pprint

if fds_path.parent == csv_path.parent:
    pprint(characterize(df, fds), sort_dicts=False)
else:
    print(
        f"skipped FDs: {fds_path.parent.name}/fds.csv and {csv_path.parent.name}/clean.csv are from different datasets"
    )
    pprint(characterize_dataset(df), sort_dicts=False)
    pprint(characterize_fds(fds), sort_dicts=False)

In [ ]:
# for a given FD, show each distinct LHS tuple once with its occurrence count,
# sorted by the LHS columns so duplicates / outliers are easy to scan.

from eval.fd_eval import fd_lhs_redundancy

red = fd_lhs_redundancy(df, fds)

def lhs_view(fd, *, n=30):
    cols = list(fd.lhs_attributes)  # always a tuple of column names
    return df.group_by(cols).len(name="count").sort(cols).head(n)

i = 7
fd = fds[i]
print(f"[{i}] {fd}   lhs_redundancy = {red[fd.lhs_attributes]:.3f}")
lhs_view(fd, n=30)